<a href="https://colab.research.google.com/github/cauarichard/crud2/blob/main/extractautomatizado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Caminho base até a pasta "videos_celeb"
BASE_DIR = "/content/drive/MyDrive/dataset/real_fake/videos_celeb"

print("Conteúdo de BASE_DIR:")
print(BASE_DIR)
print(os.listdir(BASE_DIR))


Mounted at /content/drive
Conteúdo de BASE_DIR:
/content/drive/MyDrive/dataset/real_fake/videos_celeb
['fakes', 'real']


In [2]:
import os
import cv2
import dlib

# ==============================
# CONFIGURAÇÕES GERAIS
# ==============================

# Caminho base até a pasta "videos_celeb"
BASE_DIR = "/content/drive/MyDrive/dataset/real_fake/videos_celeb"

# Dentro de BASE_DIR você tem:
#   real/id0_real, id1_real, ...
#   fakes/id0_fake, id1_fake, ...
VIDEOS_REAL_BASE = os.path.join(BASE_DIR, "real")
VIDEOS_FAKE_BASE = os.path.join(BASE_DIR, "fakes")

# Onde as faces serão salvas
FACES_REAIS_ROOT = os.path.join(BASE_DIR, "faces_reais")
FACES_FAKES_ROOT = os.path.join(BASE_DIR, "faces_fakes")

os.makedirs(FACES_REAIS_ROOT, exist_ok=True)
os.makedirs(FACES_FAKES_ROOT, exist_ok=True)

FPS_EXTRACTION = 4      # frames por segundo
MARGIN = 0.25           # margem em torno da face

# Detector de face do dlib
face_detector = dlib.get_frontal_face_detector()


# ==============================
# FUNÇÕES AUXILIARES
# ==============================

def listar_ids_disponiveis():
    """Lista IDs base (id0, id1, ...) presentes em real/ e/ou fakes/."""
    ids_real = []
    ids_fake = []

    if os.path.isdir(VIDEOS_REAL_BASE):
        ids_real = [d for d in os.listdir(VIDEOS_REAL_BASE)
                    if os.path.isdir(os.path.join(VIDEOS_REAL_BASE, d))]
    if os.path.isdir(VIDEOS_FAKE_BASE):
        ids_fake = [d for d in os.listdir(VIDEOS_FAKE_BASE)
                    if os.path.isdir(os.path.join(VIDEOS_FAKE_BASE, d))]

    # Extrai só o prefixo até o underline: "id0_real" -> "id0"
    ids_real_base = {d.split("_")[0] for d in ids_real}
    ids_fake_base = {d.split("_")[0] for d in ids_fake}

    ids_todos = sorted(ids_real_base | ids_fake_base)

    print("IDs encontrados (digite exatamente assim, ex: id0, id5):")
    for i in ids_todos:
        print("-", i)

    return ids_todos


def extract_faces_from_videos_dlib(input_folder, output_folder, label,
                                   fps_extraction=FPS_EXTRACTION, margin=MARGIN):
    """Extrai faces dos vídeos em input_folder e salva em output_folder."""
    os.makedirs(output_folder, exist_ok=True)

    videos = [f for f in os.listdir(input_folder)
              if f.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))]

    print(f"\nProcessando {len(videos)} vídeos em: {input_folder}\n")

    for video_name in videos:
        video_path = os.path.join(input_folder, video_name)
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            print(f"Erro ao abrir vídeo: {video_name}")
            continue

        video_fps = cap.get(cv2.CAP_PROP_FPS)
        if video_fps <= 0:
            # fallback para evitar divisão por zero
            video_fps = fps_extraction

        frame_interval = int(video_fps // fps_extraction) if video_fps >= fps_extraction else 1

        frame_count = 0
        saved_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            if frame_count % frame_interval == 0:
                rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                faces = face_detector(rgb, 1)

                for i, face in enumerate(faces):
                    x1 = face.left()
                    y1 = face.top()
                    x2 = face.right()
                    y2 = face.bottom()

                    w = x2 - x1
                    h = y2 - y1
                    x_margin = int(w * margin)
                    y_margin = int(h * margin)

                    x1m = max(0, x1 - x_margin)
                    y1m = max(0, y1 - y_margin)
                    x2m = min(frame.shape[1], x2 + x_margin)
                    y2m = min(frame.shape[0], y2 + y_margin)

                    face_crop = frame[y1m:y2m, x1m:x2m]

                    output_filename = f"{label}_{os.path.splitext(video_name)[0]}_frame{saved_count:05d}.jpg"
                    output_path = os.path.join(output_folder, output_filename)
                    cv2.imwrite(output_path, face_crop)
                    saved_count += 1

            frame_count += 1

        cap.release()
        print(f"Vídeo {video_name}: {saved_count} faces extraídas.")

    print(f"\nFinalizado: faces salvas em {output_folder}")


# ==============================
# BLOCO PRINCIPAL (INTERATIVO)
# ==============================

def main():
    print("Caminho base atual:", BASE_DIR)
    print("Listando IDs disponíveis...\n")
    ids_todos = listar_ids_disponiveis()

    if not ids_todos:
        print("Nenhum ID encontrado. Verifique as pastas 'real' e 'fakes'.")
        return

    celeb_id = input("\nDigite o ID (ex: id0, id5): ").strip()

    if celeb_id not in ids_todos:
        print(f"ID '{celeb_id}' não encontrado.")
        return

    # Nomes de pasta esperados: id0_real e id0_fake
    real_id_folder = f"{celeb_id}_real"
    fake_id_folder = f"{celeb_id}_fake"

    real_id_videos = os.path.join(VIDEOS_REAL_BASE, real_id_folder)
    fake_id_videos = os.path.join(VIDEOS_FAKE_BASE, fake_id_folder)

    # Pastas de faces para esse ID
    id_faces_real_dir = os.path.join(FACES_REAIS_ROOT, f"{celeb_id}_faces_real")
    id_faces_fake_dir = os.path.join(FACES_FAKES_ROOT, f"{celeb_id}_faces_fake")

    existe_real = os.path.isdir(real_id_videos)
    existe_fake = os.path.isdir(fake_id_videos)

    if not existe_real and not existe_fake:
        print(f"Nenhuma pasta encontrada para {celeb_id} em real/ ou fakes/.")
        return

    if existe_real:
        extract_faces_from_videos_dlib(real_id_videos, id_faces_real_dir, label="real")
    else:
        print(f"Nenhuma pasta REAL encontrada para {celeb_id}: {real_id_videos}")

    if existe_fake:
        extract_faces_from_videos_dlib(fake_id_videos, id_faces_fake_dir, label="fake")
    else:
        print(f"Nenhuma pasta FAKE encontrada para {celeb_id}: {fake_id_videos}")

    print(f"\n===== TUDO CONCLUÍDO — {celeb_id} PROCESSADO =====")



main()


Caminho base atual: /content/drive/MyDrive/dataset/real_fake/videos_celeb
Listando IDs disponíveis...

IDs encontrados (digite exatamente assim, ex: id0, id5):
- id0
- id1
- id10
- id11
- id12
- id13
- id16
- id17
- id19
- id2
- id20
- id21
- id22
- id23
- id24
- id25
- id26
- id27
- id28
- id29
- id3
- id30
- id31
- id32
- id33
- id34
- id35
- id36
- id37
- id38
- id39
- id4
- id40
- id41
- id42
- id43
- id44
- id45
- id46
- id47
- id48
- id49
- id5
- id50
- id51
- id52
- id53
- id54
- id55
- id56
- id57
- id58
- id59
- id6
- id60
- id61
- id7
- id8
- id9

Digite o ID (ex: id0, id5): id0

Processando 10 vídeos em: /content/drive/MyDrive/dataset/real_fake/videos_celeb/real/id0_real

Vídeo id0_0000.mp4: 67 faces extraídas.
Vídeo id0_0001.mp4: 44 faces extraídas.
Vídeo id0_0002.mp4: 50 faces extraídas.
Vídeo id0_0003.mp4: 76 faces extraídas.
Vídeo id0_0004.mp4: 47 faces extraídas.
Vídeo id0_0005.mp4: 66 faces extraídas.
Vídeo id0_0006.mp4: 77 faces extraídas.
Vídeo id0_0007.mp4: 69 faces